
# POC SJUR - Análise de Citações Judiciais

Esta POC demonstra a detecção de citações judiciais em e-mails recebidos pela FGV.

**Fluxo da POC:**
1. Inserir ou colar um texto de e-mail.
2. Processar o texto para detectar se é uma citação judicial.
3. Visualizar a classificação gerada.
4. Avaliar se a classificação está correta ou não.
5. (Opcional) Inserir uma observação sobre a avaliação.


In [ ]:
# Célula de imports no notebook
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import datetime
import re
from dotenv import load_dotenv

# Importações corrigidas
from modules import (
    highlight_text,
    count_occurrences,
    extrair_metadados,
    DeepSeekLegalClassifier,
    inicializar_base_avaliacoes,
    registrar_avaliacao
)

load_dotenv()

USE_DEEPSEEK = True
API_KEY = os.getenv("DEEPSEEK_API_KEY")

if USE_DEEPSEEK:
    classifier = DeepSeekLegalClassifier(API_KEY)

inicializar_base_avaliacoes()


In [ ]:

def classificar_texto(texto):
    if USE_DEEPSEEK:
        resultado = classifier.classify_text(texto)
        if resultado.status != "sucesso":
            return "erro"
        return resultado.classification
    else:
        return "classificação local não implementada"


In [ ]:

# Widgets de interação

texto_input = widgets.Textarea(
    value='',
    placeholder='Cole aqui o texto do e-mail para análise...',
    description='Texto:',
    layout=widgets.Layout(width='100%', height='150px')
)

botao_processar = widgets.Button(
    description='Processar Texto',
    button_style='success'
)

saida_classificacao = widgets.Output()

avaliador_concorda = widgets.RadioButtons(
    options=['Sim', 'Não'],
    description='Concorda com a classificação?',
    disabled=False
)

campo_observacao = widgets.Textarea(
    value='',
    placeholder='Digite uma observação opcional...',
    description='Observação:',
    layout=widgets.Layout(width='100%', height='100px')
)

botao_registrar = widgets.Button(
    description='Registrar Avaliação',
    button_style='info'
)

saida_avaliacao = widgets.Output()


In [ ]:
# Widgets para a funcionalidade de busca
busca_input = widgets.Text(
    value='',
    placeholder='Digite termos para buscar (separados por ";")',
    description='Buscar:',
    layout=widgets.Layout(width='80%')
)

botao_buscar = widgets.Button(
    description='Buscar',
    button_style='warning'
)

saida_busca = widgets.Output()

def ao_clicar_buscar(b):
    with saida_busca:
        clear_output()
        texto = texto_input.value.strip()
        termos = busca_input.value.strip()
        
        if not texto:
            print("⚠️ Por favor, insira um texto para análise.")
            return
            
        if not termos:
            print("⚠️ Por favor, digite termos para buscar.")
            return
            
        highlighted = highlight_text(texto, termos)
        display(widgets.HTML(f'<div style="border:1px solid #ccc; padding:10px; margin:10px 0;">{highlighted}</div>'))
        
        # Conta ocorrências usando a função do módulo
        counts = count_occurrences(texto, termos)
        
        print("\n🔍 Resultados da busca:")
        for term, count in counts.items():
            print(f" - '{term}': {count} ocorrência(s)")

botao_buscar.on_click(ao_clicar_buscar)


In [1]:

# Célula 1: Imports e configuração inicial (mantida igual)

# Célula 2: Função classificar_texto (mantida igual)

# Célula 3: Widgets de interação (mantida igual)

# Célula 4: Função de busca (mantida igual)

# Célula 5: Funções de processamento e registro (CORRIGIDA)
def ao_clicar_processar(b):
    with saida_classificacao:
        clear_output()
        texto = texto_input.value.strip()
        if not texto:
            print("⚠️ Por favor, insira um texto para análise.")
            return
            
        classificacao = classificar_texto(texto)
        metadados = extrair_metadados(texto)
        
        print(f"🔍 Classificação sugerida: {classificacao}")
        print(f"📄 Número do Processo: {metadados['Número do Processo']}")
        
        if metadados['Prazos'] != "Nenhum prazo identificado":
            print("\n⏳ Prazos encontrados:")
            for prazo in metadados['Prazos']:
                print(f"  - {prazo['texto']} ({prazo['dias']} {prazo['tipo']})")
                
        if metadados['Partes Réus'] != "Nenhuma parte ré identificada":
            print("\n⚖️ Partes Réus identificadas:")
            for i, reu in enumerate(metadados['Partes Réus'], 1):
                tipo = "(réu por ser a última parte listada)" if reu['tipo_indicador'] == "ultima_parte" else ""
                print(f"  {i}. {reu['nome']} {tipo}")

def ao_clicar_registrar(b):
    with saida_avaliacao:
        clear_output()
        texto = texto_input.value.strip()
        classificacao = classificar_texto(texto)
        metadados = extrair_metadados(texto)
        concorda = avaliador_concorda.value
        observacao = campo_observacao.value
        registrar_avaliacao(texto, classificacao, metadados, concorda, observacao)
        print("✅ Avaliação registrada com sucesso!")

# Célula 6: Conectar botões às funções
botao_processar.on_click(ao_clicar_processar)
botao_registrar.on_click(ao_clicar_registrar)
botao_buscar.on_click(ao_clicar_buscar)

# Célula 7: Exibição da interface (mantida igual)


NameError: name 'botao_registrar' is not defined

In [ ]:

# Interface Voilà

display(widgets.HTML("<h2>🔎 Insira o texto para análise:</h2>"))
display(texto_input)
display(botao_processar)
display(saida_classificacao)

display(widgets.HTML("<h2>🔍 Busca no Texto</h2>"))
display(widgets.HBox([busca_input, botao_buscar]))
display(saida_busca)

display(widgets.HTML("<h2>📝 Avaliação do Resultado:</h2>"))
display(avaliador_concorda)
display(campo_observacao)
display(botao_registrar)
display(saida_avaliacao)



# Conclusão

- Esta etapa valida o comportamento do modelo e da interface.
- As classificações são registradas com avaliação.
